In [10]:
# ============================================================
# STEP 5 - Generate Pseudo Masks for All Valid Samples
# DRIVE-SAFE STANDALONE VERSION
# ============================================================

!pip install nibabel pandas scipy tqdm -q

import os
import glob
import shutil
import numpy as np
import pandas as pd
import nibabel as nib

from tqdm import tqdm
from scipy.ndimage import (
    gaussian_filter,
    label,
    binary_opening,
    binary_closing,
    binary_fill_holes
)

# ------------------------------------------------------------
# BASE PATHS
# ------------------------------------------------------------
BASE = "/content/drive/MyDrive/MELA"

TRAIN_DIR = os.path.join(BASE, "images", "train")
VAL_DIR   = os.path.join(BASE, "images", "val")

ANN_PATH     = os.path.join(BASE, "annotations", "mela_train_val_annotations.csv")
SPACING_PATH = os.path.join(BASE, "annotations", "mela_origin_spacing.csv")

# Drive output
DRIVE_MASK_BASE_DIR  = os.path.join(BASE, "masks")
DRIVE_TRAIN_MASK_DIR = os.path.join(DRIVE_MASK_BASE_DIR, "train")
DRIVE_VAL_MASK_DIR   = os.path.join(DRIVE_MASK_BASE_DIR, "val")
DRIVE_LOG_PATH       = os.path.join(BASE, "annotations", "mela_pseudomask_log.csv")

# Local temp output
LOCAL_MASK_BASE_DIR  = "/content/mela_masks"
LOCAL_TRAIN_MASK_DIR = os.path.join(LOCAL_MASK_BASE_DIR, "train")
LOCAL_VAL_MASK_DIR   = os.path.join(LOCAL_MASK_BASE_DIR, "val")
LOCAL_LOG_PATH       = "/content/mela_pseudomask_log.csv"

os.makedirs(LOCAL_TRAIN_MASK_DIR, exist_ok=True)
os.makedirs(LOCAL_VAL_MASK_DIR, exist_ok=True)

print("LOCAL_TRAIN_MASK_DIR:", LOCAL_TRAIN_MASK_DIR)
print("LOCAL_VAL_MASK_DIR:", LOCAL_VAL_MASK_DIR)
print("DRIVE_TRAIN_MASK_DIR:", DRIVE_TRAIN_MASK_DIR)
print("DRIVE_VAL_MASK_DIR:", DRIVE_VAL_MASK_DIR)

# ------------------------------------------------------------
# LOAD CSVs
# ------------------------------------------------------------
df_ann = pd.read_csv(ANN_PATH)
df_spacing = pd.read_csv(SPACING_PATH)

df_ann.columns = [c.strip() for c in df_ann.columns]
df_spacing.columns = [c.strip() for c in df_spacing.columns]

# ------------------------------------------------------------
# BUILD FILE MAPS
# ------------------------------------------------------------
train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.nii.gz")))
val_files   = sorted(glob.glob(os.path.join(VAL_DIR, "*.nii.gz")))

train_map = {
    os.path.basename(path).replace(".nii.gz", ""): path
    for path in train_files
}

val_map = {
    os.path.basename(path).replace(".nii.gz", ""): path
    for path in val_files
}

def resolve_image_info(public_id):
    if public_id in train_map:
        return pd.Series([train_map[public_id], "train"])
    elif public_id in val_map:
        return pd.Series([val_map[public_id], "val"])
    else:
        return pd.Series([None, "missing"])

df_ann[["image_path", "split"]] = df_ann["public_id"].apply(resolve_image_info)

# ------------------------------------------------------------
# MERGE + VALID FILTER
# ------------------------------------------------------------
df_master = pd.merge(df_ann, df_spacing, on="public_id", how="left")
df_valid = df_master[df_master["split"] != "missing"].copy()
df_valid = df_valid.drop_duplicates(subset=["public_id"]).copy()

print("\nValid sample count:", len(df_valid))
print(df_valid["split"].value_counts())

# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------
def generate_pseudo_mask_for_sample(row, save_mask=True):
    public_id = row["public_id"]
    split = row["split"]
    image_path = row["image_path"]

    coordX, coordY, coordZ = int(row["coordX"]), int(row["coordY"]), int(row["coordZ"])
    lx, ly, lz = int(row["x_length"]), int(row["y_length"]), int(row["z_length"])

    result = {
        "public_id": public_id,
        "split": split,
        "image_path": image_path,
        "mask_path_local": None,
        "mask_path_drive": None,
        "method_used": None,
        "bbox_voxels": None,
        "refined_voxels": None,
        "compression_ratio": None,
        "seed_value": None,
        "seed_offset_x": None,
        "seed_offset_y": None,
        "seed_offset_z": None,
        "status": "ok",
        "error_message": None
    }

    try:
        # ----------------------------------------------------
        # LOAD IMAGE + CANONICAL
        # ----------------------------------------------------
        img = nib.load(image_path)
        img_canon = nib.as_closest_canonical(img)
        img_data = img_canon.get_fdata().astype(np.float32)

        # ----------------------------------------------------
        # TRUE CENTER IN ORIGINAL VOXEL SPACE
        # ----------------------------------------------------
        center_old_voxel = np.array([
            coordX + lx / 2.0,
            coordY + ly / 2.0,
            coordZ + lz / 2.0,
            1.0
        ], dtype=np.float32)

        center_world = img.affine @ center_old_voxel
        center_new_voxel = np.linalg.inv(img_canon.affine) @ center_world
        cx, cy, cz = center_new_voxel[:3]

        # ----------------------------------------------------
        # ROI
        # ----------------------------------------------------
        margin = 0

        x_min = max(0, int(np.floor(cx - lx / 2.0)) - margin)
        x_max = min(img_data.shape[0], int(np.ceil(cx + lx / 2.0)) + margin)

        y_min = max(0, int(np.floor(cy - ly / 2.0)) - margin)
        y_max = min(img_data.shape[1], int(np.ceil(cy + ly / 2.0)) + margin)

        z_min = max(0, int(np.floor(cz - lz / 2.0)) - margin)
        z_max = min(img_data.shape[2], int(np.ceil(cz + lz / 2.0)) + margin)

        if x_min >= x_max or y_min >= y_max or z_min >= z_max:
            raise ValueError("Invalid ROI bounds")

        roi = img_data[x_min:x_max, y_min:y_max, z_min:z_max]

        if roi.size == 0:
            raise ValueError("ROI is empty")

        # ----------------------------------------------------
        # PREPROCESS
        # ----------------------------------------------------
        roi = np.clip(roi, -200, 300)
        roi_smooth = gaussian_filter(roi, sigma=1)

        # ----------------------------------------------------
        # CENTER-GUIDED LOCAL SEED
        # ----------------------------------------------------
        center_seed_x = int(round(cx)) - x_min
        center_seed_y = int(round(cy)) - y_min
        center_seed_z = int(round(cz)) - z_min

        center_seed_x = int(np.clip(center_seed_x, 0, roi.shape[0] - 1))
        center_seed_y = int(np.clip(center_seed_y, 0, roi.shape[1] - 1))
        center_seed_z = int(np.clip(center_seed_z, 0, roi.shape[2] - 1))

        window = 8

        sx_min = max(0, center_seed_x - window)
        sx_max = min(roi.shape[0], center_seed_x + window + 1)

        sy_min = max(0, center_seed_y - window)
        sy_max = min(roi.shape[1], center_seed_y + window + 1)

        sz_min = max(0, center_seed_z - window)
        sz_max = min(roi.shape[2], center_seed_z + window + 1)

        local_patch = roi_smooth[sx_min:sx_max, sy_min:sy_max, sz_min:sz_max]

        local_idx = np.argmax(local_patch)
        lx_i, ly_i, lz_i = np.unravel_index(local_idx, local_patch.shape)

        seed_x = sx_min + lx_i
        seed_y = sy_min + ly_i
        seed_z = sz_min + lz_i

        result["seed_offset_x"] = int(seed_x - center_seed_x)
        result["seed_offset_y"] = int(seed_y - center_seed_y)
        result["seed_offset_z"] = int(seed_z - center_seed_z)

        # ----------------------------------------------------
        # THRESHOLD
        # ----------------------------------------------------
        seed_value = float(roi_smooth[seed_x, seed_y, seed_z])
        result["seed_value"] = seed_value

        delta = 18
        lower = seed_value - delta
        upper = seed_value + delta

        binary_roi = (roi_smooth >= lower) & (roi_smooth <= upper)

        # ----------------------------------------------------
        # CONNECTED COMPONENT FROM SEED
        # ----------------------------------------------------
        labeled, _ = label(binary_roi)
        seed_label = labeled[seed_x, seed_y, seed_z]

        if seed_label == 0:
            region = np.zeros_like(binary_roi, dtype=bool)
        else:
            region = (labeled == seed_label)

        # ----------------------------------------------------
        # MORPHOLOGY
        # ----------------------------------------------------
        region = binary_opening(region)
        region = binary_closing(region)
        region = binary_fill_holes(region)

        # ----------------------------------------------------
        # COMPONENT SELECTION: CENTER-NEAREST
        # ----------------------------------------------------
        labeled2, num2 = label(region)

        if num2 == 0:
            region = np.zeros_like(region, dtype=bool)
        else:
            best_label = None
            best_dist = 1e9

            for lab in range(1, num2 + 1):
                coords = np.argwhere(labeled2 == lab)
                centroid = coords.mean(axis=0)
                dist = np.linalg.norm(
                    centroid - np.array([center_seed_x, center_seed_y, center_seed_z])
                )
                if dist < best_dist:
                    best_dist = dist
                    best_label = lab

            region = (labeled2 == best_label)

        # ----------------------------------------------------
        # BUILD MASK / FALLBACK
        # ----------------------------------------------------
        min_voxels = 500
        method_used = "region_growing"

        bbox_mask_full = np.zeros_like(img_data, dtype=np.uint8)
        bbox_mask_full[x_min:x_max, y_min:y_max, z_min:z_max] = 1
        bbox_voxels = int(bbox_mask_full.sum())

        if region.sum() < min_voxels:
            method_used = "fallback_inner_box"

            shrink = 0.5
            half_x = int((lx * shrink) / 2)
            half_y = int((ly * shrink) / 2)
            half_z = int((lz * shrink) / 2)

            mask_full = np.zeros_like(img_data, dtype=np.uint8)

            fx_min = max(0, int(round(cx)) - half_x)
            fx_max = min(img_data.shape[0], int(round(cx)) + half_x)

            fy_min = max(0, int(round(cy)) - half_y)
            fy_max = min(img_data.shape[1], int(round(cy)) + half_y)

            fz_min = max(0, int(round(cz)) - half_z)
            fz_max = min(img_data.shape[2], int(round(cz)) + half_z)

            if fx_min < fx_max and fy_min < fy_max and fz_min < fz_max:
                mask_full[fx_min:fx_max, fy_min:fy_max, fz_min:fz_max] = 1
            else:
                raise ValueError("Fallback inner box invalid")
        else:
            mask_full = np.zeros_like(img_data, dtype=np.uint8)
            mask_full[x_min:x_max, y_min:y_max, z_min:z_max] = region.astype(np.uint8)

        refined_voxels = int(mask_full.sum())
        compression_ratio = float(refined_voxels) / float(bbox_voxels) if bbox_voxels > 0 else None

        result["method_used"] = method_used
        result["bbox_voxels"] = bbox_voxels
        result["refined_voxels"] = refined_voxels
        result["compression_ratio"] = compression_ratio

        # ----------------------------------------------------
        # SAVE LOCALLY ONLY
        # ----------------------------------------------------
        if save_mask:
            out_dir = LOCAL_TRAIN_MASK_DIR if split == "train" else LOCAL_VAL_MASK_DIR
            mask_path_local = os.path.join(out_dir, public_id + "_mask.nii.gz")

            mask_nifti = nib.Nifti1Image(mask_full.astype(np.uint8), img_canon.affine)
            nib.save(mask_nifti, mask_path_local)

            result["mask_path_local"] = mask_path_local

        return result

    except Exception as e:
        result["status"] = "error"
        result["method_used"] = "error"
        result["error_message"] = str(e)
        return result

# ------------------------------------------------------------
# CHOOSE SUBSET
# First test with 10, then switch to full
# ------------------------------------------------------------
USE_FULL_DATA = True

if USE_FULL_DATA:
    df_run = df_valid.copy()
else:
    df_run = df_valid.head(10).copy()

print("\nRunning on sample count:", len(df_run))

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------
results = []

for idx, (_, row) in enumerate(tqdm(df_run.iterrows(), total=len(df_run))):
    out = generate_pseudo_mask_for_sample(row, save_mask=True)
    results.append(out)

    # ara log - local
    if (idx + 1) % 5 == 0 or (idx + 1) == len(df_run):
        df_temp = pd.DataFrame(results)
        df_temp.to_csv(LOCAL_LOG_PATH, index=False)

df_results = pd.DataFrame(results)
df_results.to_csv(LOCAL_LOG_PATH, index=False)

print("\nSaved local log to:")
print(LOCAL_LOG_PATH)

# ------------------------------------------------------------
# SUMMARY (LOCAL)
# ------------------------------------------------------------
print("\n===== SUMMARY =====")
print("Total processed:", len(df_results))
print("\nStatus counts:")
print(df_results["status"].value_counts(dropna=False))

print("\nMethod counts:")
print(df_results["method_used"].value_counts(dropna=False))

print("\nSample results:")
display(df_results[[
    "public_id", "split", "method_used",
    "bbox_voxels", "refined_voxels",
    "compression_ratio", "status", "error_message"
]].head(10))

# ------------------------------------------------------------
# COPY TO DRIVE (OPTIONAL)
# ------------------------------------------------------------
COPY_TO_DRIVE = True

if COPY_TO_DRIVE:
    try:
        os.makedirs(DRIVE_TRAIN_MASK_DIR, exist_ok=True)
        os.makedirs(DRIVE_VAL_MASK_DIR, exist_ok=True)
        os.makedirs(os.path.dirname(DRIVE_LOG_PATH), exist_ok=True)

        # train masks
        for fname in os.listdir(LOCAL_TRAIN_MASK_DIR):
            src = os.path.join(LOCAL_TRAIN_MASK_DIR, fname)
            dst = os.path.join(DRIVE_TRAIN_MASK_DIR, fname)
            shutil.copy2(src, dst)

        # val masks
        for fname in os.listdir(LOCAL_VAL_MASK_DIR):
            src = os.path.join(LOCAL_VAL_MASK_DIR, fname)
            dst = os.path.join(DRIVE_VAL_MASK_DIR, fname)
            shutil.copy2(src, dst)

        # log
        shutil.copy2(LOCAL_LOG_PATH, DRIVE_LOG_PATH)

        print("\nCopied masks and log to Drive successfully.")
        print("Drive log path:", DRIVE_LOG_PATH)

    except Exception as e:
        print("\nDrive copy failed:")
        print(str(e))
        print("Local results are safe at:")
        print(" -", LOCAL_TRAIN_MASK_DIR)
        print(" -", LOCAL_VAL_MASK_DIR)
        print(" -", LOCAL_LOG_PATH)

LOCAL_TRAIN_MASK_DIR: /content/mela_masks/train
LOCAL_VAL_MASK_DIR: /content/mela_masks/val
DRIVE_TRAIN_MASK_DIR: /content/drive/MyDrive/MELA/masks/train
DRIVE_VAL_MASK_DIR: /content/drive/MyDrive/MELA/masks/val

Valid sample count: 370
split
train    260
val      110
Name: count, dtype: int64

Running on sample count: 370


100%|██████████| 370/370 [39:03<00:00,  6.33s/it]


Saved local log to:
/content/mela_pseudomask_log.csv

===== SUMMARY =====
Total processed: 370

Status counts:
status
ok    370
Name: count, dtype: int64

Method counts:
method_used
fallback_inner_box    242
region_growing        128
Name: count, dtype: int64

Sample results:


,public_id,split,method_used,bbox_voxels,refined_voxels,compression_ratio,status,error_message
0,mela_0001,train,fallback_inner_box,139934,16200,0.115769,ok,None
1,mela_0002,train,fallback_inner_box,63840,6480,0.101504,ok,None
2,mela_0003,train,region_growing,919302,70472,0.076658,ok,None
3,mela_0004,train,region_growing,1197840,36126,0.030159,ok,None
4,mela_0005,train,fallback_inner_box,127440,14560,0.114250,ok,None
5,mela_0006,train,region_growing,62016,5332,0.085978,ok,None
6,mela_0007,train,fallback_inner_box,6053780,860288,0.142108,ok,None
7,mela_0008,train,fallback_inner_box,53760,6720,0.125000,ok,None
8,mela_0009,train,fallback_inner_box,50540,5184,0.102572,ok,None
9,mela_0010,train,fallback_inner_box,6840,640,0.093567,ok,None



Copied masks and log to Drive successfully.
Drive log path: /content/drive/MyDrive/MELA/annotations/mela_pseudomask_log.csv


In [11]:
print("Toplam train mask sayısı:", len(os.listdir(TRAIN_MASK_DIR)))
print("Toplam val mask sayısı:", len(os.listdir(VAL_MASK_DIR)))

print("\nHatalı örnek sayısı:")
print((df_results["status"] == "error").sum())

print("\nFallback kullanılan örnek sayısı:")
print((df_results["method_used"] == "fallback_inner_box").sum())

print("\nRegion growing kullanılan örnek sayısı:")
print((df_results["method_used"] == "region_growing").sum())

print("\nİlk 10 sonuç:")
display(df_results[[
    "public_id", "split", "method_used",
    "bbox_voxels", "refined_voxels",
    "compression_ratio", "status"
]].head(10))

Toplam train mask sayısı: 260
Toplam val mask sayısı: 110

Hatalı örnek sayısı:
0

Fallback kullanılan örnek sayısı:
242

Region growing kullanılan örnek sayısı:
128

İlk 10 sonuç:


,public_id,split,method_used,bbox_voxels,refined_voxels,compression_ratio,status
0,mela_0001,train,fallback_inner_box,139934,16200,0.115769,ok
1,mela_0002,train,fallback_inner_box,63840,6480,0.101504,ok
2,mela_0003,train,region_growing,919302,70472,0.076658,ok
3,mela_0004,train,region_growing,1197840,36126,0.030159,ok
4,mela_0005,train,fallback_inner_box,127440,14560,0.114250,ok
5,mela_0006,train,region_growing,62016,5332,0.085978,ok
6,mela_0007,train,fallback_inner_box,6053780,860288,0.142108,ok
7,mela_0008,train,fallback_inner_box,53760,6720,0.125000,ok
8,mela_0009,train,fallback_inner_box,50540,5184,0.102572,ok
9,mela_0010,train,fallback_inner_box,6840,640,0.093567,ok
